####Построение прогнозных моделей: ARIMA и SARIMAX
####Реализация через MLE (scipy.optimize) без statsmodels.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, stats

In [2]:
import os, shutil
os.makedirs("/Users/maksimtimosuk/Desktop/models", exist_ok=True)
os.chdir("/Users/maksimtimosuk/Desktop/models")

In [3]:
df = pd.read_csv("retail_sales_mock_data.csv", parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)
df.set_index("Date", inplace=True)

sales = df["SalesAmount"].values.astype(float)
exog_promo = df["Promotion"].values.astype(float)
exog_hol = df["HolidayMonth"].values.astype(float)
dates = df.index
n = len(sales)

# Разбивка: 80% train, 20% test
split = int(n * 0.80)
y_train = sales[:split]
y_test = sales[split:]
x_train = np.column_stack([exog_promo[:split], exog_hol[:split]])
x_test = np.column_stack([exog_promo[split:], exog_hol[split:]])
dates_train = dates[:split]
dates_test = dates[split:]

print(f"Обучающая выборка: {len(y_train)} наблюдений  ({dates[0].strftime('%Y-%m')} — {dates[split-1].strftime('%Y-%m')})")
print(f"Тестовая выборка:  {len(y_test)} наблюдений  ({dates[split].strftime('%Y-%m')} — {dates[-1].strftime('%Y-%m')})")

Обучающая выборка: 38 наблюдений  (2020-01 — 2023-02)
Тестовая выборка:  10 наблюдений  (2023-03 — 2023-12)


Вспомогательные функции

In [4]:
def difference(series, d=1, D=0, s=12):
    #Сезонное и обычное дифференцирование.
    #Порядок: сначала сезонное (D раза), потом обычное (d раза).
    x = series.copy()
    for _ in range(D):
        x = x[s:] - x[:-s]
    for _ in range(d):
        x = np.diff(x)
    return x

def undifference(diff_series, original, d=1, D=0, s=12):
    #Обратное интегрирование для получения прогноза в исходном масштабе.
    x = diff_series.copy()
    # обратное обычное интегрирование
    for _ in range(d):
        init = original[len(original) - len(x) - 1]
        x = np.r_[init, x].cumsum()[1:]
    # обратное сезонное интегрирование
    for _ in range(D):
        # нужно «посеять» значения из хвоста оригинала
        seed = original[len(original) - len(x) - s: len(original) - len(x)]
        result = np.zeros(len(x))
        for i in range(len(x)):
            if i < s:
                result[i] = x[i] + seed[i]
            else:
                result[i] = x[i] + result[i - s]
        x = result
    return x

def make_lag_matrix(y, ar_lags, ma_lags=None):
    #Формирует матрицу авторегрессионных лагов.
    #Для МА лаги — по остаткам (передаётся снаружи).
    pass  # будет использовано внутри LogLik

def arima_loglik(params, y, p, q, include_const=True):
    #Условная log-likelihood для ARIMA(p,0,q) через рекурсию инноваций.
    #Параметры: [const?, phi_1..phi_p, theta_1..theta_q, log_sigma]
    idx = 0
    if include_const:
        mu = params[idx]; idx += 1
    else:
        mu = 0.0
    phi   = params[idx: idx + p];     idx += p
    theta = params[idx: idx + q];     idx += q
    log_sigma = params[idx]
    sigma2 = np.exp(2 * log_sigma)

    n_y   = len(y)
    max_lag = max(p, q, 1)
    resids = np.zeros(n_y)

    for t in range(n_y):
        pred = mu
        for i, phi_i in enumerate(phi):
            if t - i - 1 >= 0:
                pred += phi_i * y[t - i - 1]
        for j, th_j in enumerate(theta):
            if t - j - 1 >= 0:
                pred += th_j * resids[t - j - 1]
        resids[t] = y[t] - pred

    # Логарифм функции правдоподобия (нормальное распределение ошибок)
    ll = -0.5 * n_y * np.log(2 * np.pi * sigma2) \
         - 0.5 * np.sum(resids ** 2) / sigma2
    return -ll  # минимизируем отрицательный LogLik


def sarima_loglik(params, y, p, q, P, Q, s, include_const=True, exog=None):
    #SARIMAX: SARIMA(p,0,q)(P,0,Q)[s] + экзогенные переменные.
    #Параметры: [const?, phi_1..p, theta_1..q, Phi_1..P, Theta_1..Q, beta (exog), log_sigma]
    idx = 0
    if include_const:
        mu = params[idx]; idx += 1
    else:
        mu = 0.0
    phi = params[idx: idx + p]; idx += p
    theta = params[idx: idx + q]; idx += q
    PHI = params[idx: idx + P]; idx += P
    THETA = params[idx: idx + Q]; idx += Q
    n_exog = exog.shape[1] if exog is not None else 0
    beta = params[idx: idx + n_exog]; idx += n_exog
    log_sigma = params[idx]
    sigma2 = np.exp(2 * log_sigma)

    n_y = len(y)
    resids = np.zeros(n_y)

    for t in range(n_y):
        pred = mu
        # AR часть
        for i, phi_i in enumerate(phi):
            if t - i - 1 >= 0:
                pred += phi_i * y[t - i - 1]
        # MA часть
        for j, th_j in enumerate(theta):
            if t - j - 1 >= 0:
                pred += th_j * resids[t - j - 1]
        # Сезонный AR
        for I, PHI_i in enumerate(PHI):
            lag = (I + 1) * s
            if t - lag >= 0:
                pred += PHI_i * y[t - lag]
        # Сезонный MA
        for J, TH_j in enumerate(THETA):
            lag = (J + 1) * s
            if t - lag >= 0:
                pred += TH_j * resids[t - lag]
        # Экзогенные
        if exog is not None:
            pred += np.dot(beta, exog[t])
        resids[t] = y[t] - pred

    ll = -0.5 * n_y * np.log(2 * np.pi * sigma2) \
         - 0.5 * np.sum(resids ** 2) / sigma2
    return -ll

def fit_arima(y, p, d, q, s=1, D=0, include_const=True):
    #Обёртка: дифференцировать → оптимизировать → вернуть результат
    y_diff = difference(y, d=d, D=D, s=s)
    n_params = (1 if include_const else 0) + p + q + 1  # +1 для log_sigma
    x0 = np.zeros(n_params)
    x0[-1] = np.log(np.std(y_diff) + 1e-8)

    res = optimize.minimize(
        arima_loglik, x0,
        args=(y_diff, p, q, include_const),
        method="Nelder-Mead",
        options={"maxiter": 5000, "xatol": 1e-6, "fatol": 1e-6})
    return res, y_diff

def fit_sarimax(y, p, d, q, P, D, Q, s=12, include_const=True, exog=None):
    #Обёртка для SARIMAX
    y_diff = difference(y, d=d, D=D, s=s)
    # при дифференцировании exog нужно обрезать соответственно
    n_cut = len(y) - len(y_diff)
    exog_diff = exog[n_cut:] if exog is not None else None

    n_exog   = exog_diff.shape[1] if exog_diff is not None else 0
    n_params = (1 if include_const else 0) + p + q + P + Q + n_exog + 1
    x0 = np.zeros(n_params)
    x0[-1] = np.log(np.std(y_diff) + 1e-8)

    res = optimize.minimize(
        sarima_loglik, x0,
        args=(y_diff, p, q, P, Q, s, include_const, exog_diff),
        method="Nelder-Mead",
        options={"maxiter": 8000, "xatol": 1e-6, "fatol": 1e-6})
    return res, y_diff, n_cut

def get_residuals_arima(params, y_diff, p, q, include_const=True):
    #Вычислить остатки из оптимальных параметров ARIMA
    idx = 0
    mu    = params[idx] if include_const else 0.0; idx += include_const
    phi   = params[idx: idx + p];     idx += p
    theta = params[idx: idx + q];     idx += q
    n_y   = len(y_diff)
    resids = np.zeros(n_y)
    for t in range(n_y):
        pred = mu
        for i, phi_i in enumerate(phi):
            if t - i - 1 >= 0: pred += phi_i * y_diff[t - i - 1]
        for j, th_j in enumerate(theta):
            if t - j - 1 >= 0: pred += th_j * resids[t - j - 1]
        resids[t] = y_diff[t] - pred
    return resids

def get_residuals_sarima(params, y_diff, p, q, P, Q, s=12,
                          include_const=True, exog=None):
    #Вычислить остатки SARIMAX
    idx = 0
    mu = params[idx] if include_const else 0.0; idx += include_const
    phi = params[idx: idx + p];   idx += p
    theta = params[idx: idx + q];   idx += q
    PHI = params[idx: idx + P];   idx += P
    THETA = params[idx: idx + Q];   idx += Q
    n_exog = exog.shape[1] if exog is not None else 0
    beta = params[idx: idx + n_exog]; idx += n_exog
    n_y = len(y_diff)
    resids = np.zeros(n_y)
    for t in range(n_y):
        pred = mu
        for i, phi_i in enumerate(phi):
            if t-i-1 >= 0: pred += phi_i * y_diff[t-i-1]
        for j, th_j in enumerate(theta):
            if t-j-1 >= 0: pred += th_j * resids[t-j-1]
        for I, PHI_i in enumerate(PHI):
            lag = (I+1)*s
            if t-lag >= 0: pred += PHI_i * y_diff[t-lag]
        for J, TH_j in enumerate(THETA):
            lag = (J+1)*s
            if t-lag >= 0: pred += TH_j * resids[t-lag]
        if exog is not None: pred += np.dot(beta, exog[t])
        resids[t] = y_diff[t] - pred
    return resids

def forecast_arima(y_train_diff, params, p, q, h, include_const=True):
    #h-шаговый прогноз ARIMA в пространстве дифференцированного ряда
    idx = 0
    mu    = params[idx] if include_const else 0.0; idx += include_const
    phi   = params[idx: idx + p]; idx += p
    theta = params[idx: idx + q]

    # вычислим остатки на обучающей выборке
    resids_hist = get_residuals_arima(params, y_train_diff, p, q, include_const)
    y_ext  = np.concatenate([y_train_diff, np.zeros(h)])
    e_ext  = np.concatenate([resids_hist, np.zeros(h)])
    n_hist = len(y_train_diff)

    for t in range(n_hist, n_hist + h):
        pred = mu
        for i, phi_i in enumerate(phi):
            if t-i-1 >= 0: pred += phi_i * y_ext[t-i-1]
        for j, th_j in enumerate(theta):
            if t-j-1 >= 0: pred += th_j * e_ext[t-j-1]
        y_ext[t] = pred
    return y_ext[n_hist:]

def forecast_sarima(y_train_diff, params, p, q, P, Q, s, h, include_const=True, exog_future=None):
    #h-шаговый прогноз SARIMAX
    idx = 0
    mu = params[idx] if include_const else 0.0; idx += include_const
    phi = params[idx: idx + p]; idx += p
    theta = params[idx: idx + q]; idx += q
    PHI = params[idx: idx + P]; idx += P
    THETA = params[idx: idx + Q]; idx += Q
    n_exog = 0
    if exog_future is not None: n_exog = exog_future.shape[1]
    beta = params[idx: idx + n_exog]

    resids_hist = get_residuals_sarima(params, y_train_diff, p, q, P, Q, s,include_const,exog_future[:len(y_train_diff)]if exog_future is not None else None)
    y_ext = np.concatenate([y_train_diff, np.zeros(h)])
    e_ext = np.concatenate([resids_hist, np.zeros(h)])
    n_hist = len(y_train_diff)

    for t in range(n_hist, n_hist + h):
        pred = mu
        for i, phi_i in enumerate(phi):
            if t-i-1 >= 0: pred += phi_i * y_ext[t-i-1]
        for j, th_j in enumerate(theta):
            if t-j-1 >= 0: pred += th_j * e_ext[t-j-1]
        for I, PHI_i in enumerate(PHI):
            lag = (I+1)*s
            if t-lag >= 0: pred += PHI_i * y_ext[t-lag]
        for J, TH_j in enumerate(THETA):
            lag = (J+1)*s
            if t-lag >= 0: pred += TH_j * e_ext[t-lag]
        if exog_future is not None:
            future_idx = t - n_hist
            pred += np.dot(beta, exog_future[len(y_train_diff) + future_idx])
        y_ext[t] = pred
    return y_ext[n_hist:]

def info_criteria(neg_ll, n_obs, n_params):
    #AIC и BIC из neg. log-likelihood
    ll   = -neg_ll
    aic  = -2 * ll + 2 * n_params
    bic  = -2 * ll + n_params * np.log(n_obs)
    return aic, bic

Подбор параметров (на основе EDA и декомпозиции)

In [5]:
# ACF/PACF из скрипта 1 показали:
# Лаг 1 (PACF значим) → p = 1
# Лаг 12 (ACF значим) → сезонный P = 1
# Ряд близок к стационарному по уровню, но сезонный тренд есть → d=0, D=1 (одно сезонное различие)

# Рассмотрим два варианта:
# M1: ARIMA(1, 1, 1) — простая модель без сезонной части
# M2: SARIMA(1, 1, 0)(1, 1, 0)[12] с экзогенными (SARIMAX)

In [6]:
#ОБУЧЕНИЕ МОДЕЛИ M1: ARIMA(1,1,1)
res_arima, y_diff_arima = fit_arima(y_train, p=1, d=1, q=1, include_const=True)
print(f"{res_arima.message}")
print(f"-LogLik = {res_arima.fun:.4f}")
n_params_m1 = 1 + 1 + 1 + 1  # const + phi + theta + log_sigma
aic_m1, bic_m1 = info_criteria(res_arima.fun, len(y_diff_arima), n_params_m1)
print(f"AIC = {aic_m1:.2f},BIC = {bic_m1:.2f}")

# Прогноз M1 на тестовой выборке
h = len(y_test)
fc_diff_m1 = forecast_arima(y_diff_arima, res_arima.x, p=1, q=1, h=h)
# Обратное интегрирование: d=1
fc_m1 = np.r_[y_train[-1], fc_diff_m1].cumsum()[1:]

print("\nПрогноз M1:")
for i, (fc, act) in enumerate(zip(fc_m1, y_test)):
    print(f"t+{i+1:02d}: прогноз={fc:8.1f}  факт={act:8.1f}  "f"ошибка={act - fc:+7.1f}")


#ОБУЧЕНИЕ МОДЕЛИ M2: SARIMAX(1,1,0)(1,1,0)[12]

# Экзогенные переменные: Promotion + HolidayMonth
exog_full = np.column_stack([exog_promo, exog_hol])
res_sarima, y_diff_sarima, n_cut_sarima = fit_sarimax(
    y_train, p=1, d=1, q=0, P=1, D=1, Q=0, s=12,
    include_const=True,
    exog=exog_full[:split])
print(f"{res_sarima.message}")
print(f"-LogLik = {res_sarima.fun:.4f}")

# Параметры: const + phi_1 + PHI_1 + beta_promo + beta_hol + log_sigma
n_params_m2 = 1 + 1 + 0 + 1 + 0 + 2 + 1
aic_m2, bic_m2 = info_criteria(res_sarima.fun, len(y_diff_sarima), n_params_m2)
print(f"AIC = {aic_m2:.2f},BIC = {bic_m2:.2f}")

# Прогноз M2 — нужен exog на всём промежутке (train + test)
exog_full_ahead = np.column_stack([exog_promo, exog_hol])
n_cut = n_cut_sarima  # сколько наблюдений потеряно при дифференцировании

fc_diff_m2 = forecast_sarima(y_diff_sarima, res_sarima.x, p=1, q=0, P=1, Q=0, s=12, h=h,
    include_const=True, exog_future=exog_full_ahead[n_cut:])

# Обратное интегрирование: d=1, D=1, s=12
# Для этого нужны «семена» из конца train
def undiff_d1_D1(fc_diff, y_original, s=12, d=1, D=1):
    #Обратное интегрирование для d=1, D=1
    # Сначала обратное d
    init_val = y_original[-1] - y_original[-1-s]# последнее сезонно-дифф значение
    # У нас d=1 из сезонно-дифф ряда
    tmp = np.zeros(len(fc_diff))
    seed_d = y_original[-s] - y_original[-s-s] if len(y_original) > 2*s else 0
    # простое cumsum от последнего значения дифф ряда
    last_sdiff = (y_original[-1] - y_original[-1-s])# последний сезонно-дифф elem
    tmp[0] = last_sdiff + fc_diff[0]
    for i in range(1, len(fc_diff)):
        tmp[i] = tmp[i-1] + fc_diff[i]
    # Теперь обратное D (сезонное)
    result = np.zeros(len(tmp))
    seed_vals = y_original[-(s):]   # последние s наблюдений
    for i in range(len(tmp)):
        if i < s:
            result[i] = tmp[i] + seed_vals[i]
        else:
            result[i] = tmp[i] + result[i-s]
    return result
fc_m2 = undiff_d1_D1(fc_diff_m2, y_train, s=12, d=1, D=1)

print("\nПрогноз M2:")
for i, (fc, act) in enumerate(zip(fc_m2, y_test)):
    print(f"t+{i+1:02d}: прогноз={fc:8.1f}  факт={act:8.1f}  "f"ошибка={act - fc:+7.1f}")

Optimization terminated successfully.
-LogLik = 335.3167
AIC = 678.63,BIC = 685.08

Прогноз M1:
t+01: прогноз= 13447.8  факт= 13534.0  ошибка=  +86.2
t+02: прогноз= 13829.1  факт= 12048.0  ошибка=-1781.1
t+03: прогноз= 13509.1  факт= 12949.0  ошибка= -560.1
t+04: прогноз= 13778.4  факт=  9104.0  ошибка=-4674.4
t+05: прогноз= 13552.6  факт= 10042.0  ошибка=-3510.6
t+06: прогноз= 13742.8  факт= 11566.0  ошибка=-2176.8
t+07: прогноз= 13583.4  факт= 11759.0  ошибка=-1824.4
t+08: прогноз= 13717.8  факт= 11890.0  ошибка=-1827.8
t+09: прогноз= 13605.3  факт= 11770.0  ошибка=-1835.3
t+10: прогноз= 13700.3  факт= 17996.0  ошибка=+4295.7
Optimization terminated successfully.
-LogLik = 224.3834
AIC = 460.77,BIC = 468.08

Прогноз M2:
t+01: прогноз= 11389.5  факт= 13534.0  ошибка=+2144.5
t+02: прогноз= 10570.4  факт= 12048.0  ошибка=+1477.6
t+03: прогноз=  7933.0  факт= 12949.0  ошибка=+5016.0
t+04: прогноз=  7006.0  факт=  9104.0  ошибка=+2098.0
t+05: прогноз=  8078.3  факт= 10042.0  ошибка=+1963.

Долгосрочный прогноз (максимально возможный горизонт)

In [7]:
h_long = 12  # прогноз на год вперёд
future_dates = pd.date_range(dates[-1] + pd.offsets.MonthBegin(1), periods=h_long, freq="MS")
future_promo  = np.zeros(h_long)
future_holiday = np.array([1 if d.month == 12 else 0 for d in future_dates])
exog_future_long = np.column_stack([future_promo, future_holiday])

exog_long = np.vstack([exog_full, exog_future_long])

# Полный горизонт = тест (h) + долгосрочный (h_long)
h_total = h + h_long
# exog_long содержит train+test+future; нужен кусок начиная с n_cut
exog_for_long = exog_long[n_cut_sarima: n_cut_sarima + len(y_diff_sarima) + h_total]

fc_diff_long_m2 = forecast_sarima(
    y_diff_sarima, res_sarima.x,
    p=1, q=0, P=1, Q=0, s=12, h=h_total,
    include_const=True,
    exog_future=exog_for_long)
fc_long_m2_raw = undiff_d1_D1(fc_diff_long_m2, y_train, s=12)
# интересует только хвост — долгосрочный прогноз (после тестового горизонта)
fc_long_m2 = fc_long_m2_raw[h:]

Визуализация прогнозов

In [8]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
fig.suptitle("Прогнозы моделей ARIMA и SARIMAX", fontsize=13)

# --- M1: ARIMA ---
ax = axes[0]
ax.plot(dates_train, y_train, color="#2c7bb6", lw=1.5, label="Обучение")
ax.plot(dates_test,  y_test,  color="black",   lw=1.5, label="Факт (тест)")
ax.plot(dates_test,  fc_m1,   color="#d7191c", lw=2, linestyle="--",
        marker="o", ms=5, label="ARIMA(1,1,1) прогноз")
ax.fill_between(dates_test,
                fc_m1 * 0.90, fc_m1 * 1.10,
                alpha=0.15, color="#d7191c", label="±10% коридор")
ax.set_title("M1: ARIMA(1,1,1)")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylabel("SalesAmount")

# --- M2: SARIMAX + долгосрочный ---
ax = axes[1]
ax.plot(dates_train, y_train, color="#2c7bb6", lw=1.5, label="Обучение")
ax.plot(dates_test,  y_test,  color="black",   lw=1.5, label="Факт (тест)")
ax.plot(dates_test,  fc_m2,   color="#1a9641", lw=2, linestyle="--",
        marker="s", ms=5, label="SARIMAX(1,1,0)(1,1,0)[12] тест")
ax.plot(future_dates, fc_long_m2, color="#7b3294", lw=2, linestyle="-.",
        marker="^", ms=5, label="SARIMAX долгосрочный прогноз (+12 мес.)")
ax.fill_between(future_dates,
                fc_long_m2 * 0.88, fc_long_m2 * 1.12,
                alpha=0.15, color="#7b3294", label="±12% коридор")
ax.axvline(dates[-1], color="gray", linestyle=":", lw=1.5)
ax.set_title("M2: SARIMAX(1,1,0)(1,1,0)[12] + долгосрочный прогноз")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylabel("SalesAmount")

plt.tight_layout()
plt.savefig("models_forecast.png", dpi=150, bbox_inches="tight")
plt.close()

In [9]:
np.save("arima_params.npy", res_arima.x)
np.save("sarima_params.npy", res_sarima.x)
np.save("y_train.npy", y_train)
np.save("y_test.npy", y_test)
np.save("fc_m1.npy", fc_m1)
np.save("fc_m2.npy", fc_m2)
np.save("y_diff_arima.npy", y_diff_arima)
np.save("y_diff_sarima.npy", y_diff_sarima)
np.save("exog_full.npy", exog_full)
np.save("n_cut_sarima.npy", np.array([n_cut_sarima]))
info = np.array([aic_m1, bic_m1, aic_m2, bic_m2,
                 res_arima.fun, res_sarima.fun,
                 n_params_m1, n_params_m2,
                 len(y_diff_arima), len(y_diff_sarima)])
np.save("model_info.npy", info)